# Lab 8: Report

## Charpy Impact Testing

The main objective of this lab is to understand and explain fracture under severe loading conditions (high loading rate or extreme temperatures). In this notebook you will:

- Plot the impact energy of four metals as a function of testing temperature.
- Determine the ductile-to-brittle transition temperature (DBTT) of the two steels.
- Describe how the fracture mode of each material changes with temperature.

Recommended reading: Callister, *Materials Science and Engineering: An Introduction*, Section 8.6.

## 1. Enter the data into the notebook

In Lab 7 you imported a large `.csv` file with code. This lab has only a few measurements, so this time you will **type the data into a pandas DataFrame by hand**.

1. Open `ImpactEnergy.csv` (in Excel or a text editor) and inspect it.
    - Each row is one testing temperature.
    - Each material has its own column of impact energies.
    - Not every material was tested at every temperature. The steels were tested at 0 °C and the brass and aluminum at room temperature (RT, 20 °C).
1. Note the **units** of the temperatures and the impact energies, and the **calibration value** at the bottom of the file.

Run the cell below to import the necessary libraries.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from scipy.optimize import curve_fit

### 1.1 Building a DataFrame by hand

A DataFrame can be built from a **dictionary**, written inside curly brackets `{ }`:

- each **key** (the text before the colon) becomes a column name;
- each **value** (after the colon) is a list, inside square brackets `[ ]`, holding that column's entries from top to bottom;
- every list must have the **same length**, because each position in the lists is one row of the table;
- use `np.nan` ("not a number") for an empty cell, i.e. a measurement that was not made.

Run the example below and compare the code with the table it produces.

In [ ]:
# Example: a small table of made-up measurements
example = pd.DataFrame({
    "Time (s)": [0, 10, 20, 30],
    "Length (mm)": [50.0, 50.4, np.nan, 51.1],     # no measurement was made at 20 s
})

display(example)

### 1.2 Enter your impact data

Complete the code below with the values from `ImpactEnergy.csv`.

- Make one row for **every** testing temperature in the file, in the same order for every column.
- Enter the impact energies **exactly as recorded**, in ft-lb. You will convert the units in Section 2.
- Use `np.nan` wherever a material was not tested at that temperature.
- Enter the calibration value from the bottom of the file in `CALIBRATION_FTLB`.

In [ ]:
# STUDENT: enter the data from ImpactEnergy.csv. Temperatures in °C, impact energies in ft-lb.
impactDF = pd.DataFrame({
    "T (C)":         [],
    "1018 steel":    [],
    "1045 steel":    [],
    "2024 aluminum": [],
    "70/30 brass":   [],
})

# STUDENT: enter the calibration value of the impact tester in ft-lb
CALIBRATION = 0

# shift all the data by the calibration value
impactDF.iloc[:, 1:5] = impactDF.iloc[:, 1:5] - CALIBRATION

display(impactDF)

## 2. Plot impact energy versus testing temperature (20 points)

First, set your plotting choices. Give each of the four materials a unique color, marker, line style, or a combination. The names must match the column names in `MATERIALS`.

In [4]:
# STUDENT: define a color, marker and line style for each material. They are all the same by default
COLOR = {
    "1018 steel": "",
    "1045 steel": "",
    "2024 aluminum": "",
    "70/30 brass": "",
}
MARKER = {
    "1018 steel": "",
    "1045 steel": "",
    "2024 aluminum": "",
    "70/30 brass": "",
}
LINESTYLE = {
    "1018 steel": "",
    "1045 steel": "",
    "2024 aluminum": "",
    "70/30 brass": "",
}


### Plot the impact energy of all materials as a function of the testing temperature

**Variables available inside the loop** (for one material at a time):

- `data["T (C)"]`: x values
- `data[material]`: y values

`.dropna()` removes the temperatures at which a material was not tested, so each line connects all of that material's data points.



In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ftlb_to_J = YOUR RESPONSE # conversion factor to convert from Joules to ft-lb

for material in impactDF.iloc[:, 1:5]:
    # keep only the temperatures at which this material was tested
    data = impactDF[["T (C)", material]].dropna()

    # STUDENT: plot impact energy vs temperature. Use the color, marker and line style defined previously
    ax.plot(
        YOUR RESPONSE
    )


# STUDENT: include the axes labels, title, grid, and legend

plt.tight_layout()
plt.show()

# STUDENT: add a descriptive figure caption. Explain the trends in the caption.
print("Figure 1. YOUR RESPONSE")

## 3. Ductile-to-brittle transition temperature of the steels (15 points)

The handout notes that the impact-energy curves of the steels should be S-shaped and are usually fitted with a **sigmoidal function**.

A common choice for Charpy data is the hyperbolic tangent. This fit is recommended by NIST:

$$
E(T) = \frac{E_{upper}+E_{lower}}{2} +\frac{E_{upper}-E_{lower}}{2} \cdot tanh(\frac{T-DBTT}{A})
$$

- $E(T)$: impact energy at testing temperature $T$
- $E_{upper}$: the **upper shelf** energy (ductile behavior at high temperature)
- $E_{lower}$: the **lower shelf** energy (brittle behavior at low temperature)
- $DBTT$: the temperature at the center of the transition
- $A$: corresponds to the half-width of the transition region 

### 3.1 Determine the steel ductile-to-brittle transition temperatures

The `curve_fit` method from the `scipy.optimize` module allows the user to fit `x_data` and `y_data` with user defined function. Define `Sigmoid` below to determine the fitting parameters.

Use Figure 1 to make reasonable estimates of $E_{upper}$, $E_{lower}$, $DBTT$ and $A$ and list them in the `INITIAL_GUESS`variable.

For datasets with fewer than 8 datapoints NIST recommends fitting only the values of DBTT and A. The `lower shelf` and `upper shelf` values will be used directly from your input.

In [ ]:
fit_parameters = {}

INITIAL_GUESS = {
    "1018 steel": {"lower shelf": YOUR RESPONSE, "upper shelf": YOUR RESPONSE, "DBTT": YOUR RESPONSE, "A": YOUR RESPONSE},
    "1045 steel": {"lower shelf": YOUR RESPONSE, "upper shelf": YOUR RESPONSE, "DBTT": YOUR RESPONSE, "A": YOUR RESPONSE},
                 }

# STUDENT: Define the sigmoid function from above
def sigmoid(T, E_lower, E_upper, DBTT, A):
    return YOUR RESPONSE

STEELS = list(INITIAL_GUESS)

# determine the sigmoidal fit parameters for each steel
for steel in STEELS:
    data = impactDF[["T (C)", steel]].dropna()
    guess = INITIAL_GUESS[steel]
    E_lower = guess["lower shelf"]
    E_upper = guess["upper shelf"]

    def fixed_shelves_sigmoid(T, DBTT, A):
        return sigmoid(T, E_lower, E_upper, DBTT, A)

    # determine the values of DBTT and A with the curve_fit function
    fitted, _ = curve_fit( fixed_shelves_sigmoid, data["T (C)"], data[steel],  p0=[guess["DBTT"], guess["A"]],  maxfev=10000 )
    E_fit = sigmoid(fitted[0], E_lower, E_upper, *fitted)
    
    # Keep all four parameters so the plotting cell can use sigmoid(T, *params).
    fit_parameters[steel] = np.array([E_lower, E_upper, *fitted, E_fit])

# store the values in a pandas dataframe
fit_table = pd.DataFrame(
    fit_parameters,
    index=["E_lower / ft-lb", "E_upper / ft-lb", "DBTT (fitted) / °C", "A / °C⁻¹", "E_DBTT (fitted) / ftlb" ],
).T

display(fit_table.round(2))

Next, use the fitted curve to estimate the temperature at which each steel has a fracture energy of 20 Joules.

In [ ]:
# Find the temperature where each fitted steel curve reaches 20 J.
target_ftlb = 20 * ftlb_to_J
results = {}

T_smooth = np.linspace(-200, 160, 400)     # temperatures for smooth fitted curves

# Determine the temperature at which each steel has a fracture energy of 20 Joules
for steel in STEELS:
    fit = sigmoid(T_smooth, *fit_parameters[steel][0:4])
    index = (np.abs(fit - target_ftlb)).argmin()
    results[steel] = T_smooth[index]

# append the 20 Joule fracture energy temperatures to the dataframe
fit_table["DBTT (E = 20 J) / °C"] = pd.Series(results)
display(fit_table.round(2))

### 3.2 Plot the Sigmoidal fits and estimated DBTT's

Run the code below to plot the steel data together with the fitted curves and the two DBTT estimates. Complete the title, axis labels and caption.

In [ ]:

fig, ax = plt.subplots(figsize=(9, 6))

for steel in STEELS:
    data = impactDF[["T (C)", steel]].dropna()
    row = fit_table.loc[steel]
    
    # measured data
    ax.plot(data["T (C)"],
            data[steel],
            color=COLOR[steel],
            marker=MARKER[steel],
            linestyle="none",
            label=f"{steel} data",
            )

    # fitted sigmoid curve
    ax.plot(T_smooth,
            sigmoid(T_smooth, *fit_parameters[steel][0:4]),
            color=COLOR[steel],
            linestyle=LINESTYLE[steel],
            label=f"{steel} sigmoid fit",
            )

    # DBTT estimates: at 20 J (filled star) and at the midpoint energy (open star)
    ax.plot(row["DBTT (E = 20 J) / °C"],
            20*ftlb_to_J,
            marker="*",
            markersize=16,
            color=COLOR[steel],
            linestyle="none",
            label=f"{steel} DBTT at 20 J",
            )

    ax.plot(row["DBTT (fitted) / °C"],
            row["E_DBTT (fitted) / ftlb"],
            marker="*",
            markersize=16,
            markerfacecolor="white",
            color=COLOR[steel],
            linestyle="none",
            label=f"{steel} DBTT at midpoint",
            )   

# STUDENT: include the axes labels, title, grid, and legend

plt.tight_layout()
plt.show()

# STUDENT: add a descriptive figure caption
print("Figure 2. YOUR RESPONSE")

**`YOUR RESPONSE`**: list the temperature range over which each steel transitions from ductile to brittle behavior and temperature at which each steel has a fracture energy of 20 Joules.

| Steel | Transition temperature range / °C | DBTT at 20 J / °C | DBTT at Midpoint |
|:---|:---:|:---:|:---:|
| 1018 steel | YOUR RESPONSE | YOUR RESPONSE | YOUR RESPONSE |
| 1045 steel | YOUR RESPONSE | YOUR RESPONSE | YOUR RESPONSE |



## 4. DBTT from the average energy (10 points)

**`YOUR RESPONSE`**: Does the DBTT change significantly when you choose the average between the high and the low impact energy levels instead of 20 J as the value to determine the DBTT? Why or why not?

*Replace this text with your response in paragraph form.*

## 5. Fracture surface micrographs and fracture mode (20 points)

**`YOUR RESPONSE`**: Describe and comment on the changes in fracture mode of the specimens for all materials.

Refer to macroscopic deformation observations, macroscopic fracture surface images (as available), and SEM micrographs available on D2L. Call out specific images by their labels, and relate the fracture mode you see to the impact energies in Figure 1.

*Replace this text with your response in paragraph form.*


In [ ]:
# Use this code cell to display the relevant images to support your discussion
# Refer to the lab7 report to see how to include images in your jupyter notebook
# # Remember to include captions for each figure

## CONTRIBUTION STATEMENT

*List each contributor and what they did.*

**`YOUR RESPONSE`**

| Contributor | Contribution |
|:---|:---|
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |

## SUBMISSION

When this notebook is complete and ready to submit, export it as a PDF file. If the PDF compile fails check your $\LaTeX$ code for errors.